# ML-HW10: RAG по резюме

Локальная RAG: FAISS + `qwen3:4b-instruct` (Ollama), 18 обезличенных резюме, две базы (с тегами / без), эксперименты с параметрами поиска и генерации.


## 1. Импорты

In [1]:
import os
from pathlib import Path

os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TQDM_DISABLE'] = '1'

import pandas as pd
from IPython.display import display

from rag_system import (
    CACHE_DIR,
    GenParams,
    OLLAMA_MODEL,
    DATA_CSV,
    LAB_DATA_DIR,
    SELECTED_CSV,
    ensure_resume_indices,
    load_resume_documents,
    rag_answer,
)

print('модель:', OLLAMA_MODEL)
print('данные:', DATA_CSV)


модель: qwen3:4b-instruct
данные: C:\Users\Zhenya\Desktop\МО 3\mo3_team_colabs\labs\ML-HW10\data\resumes_selected.csv


## 2. Данные

18 резюме (по 3 на категорию) в `data/resumes_selected.csv`. В индекс — сырой текст с меткой `Кандидат №N`.


In [2]:
print(LAB_DATA_DIR)
print(SELECTED_CSV.exists())

docs_tags, df = load_resume_documents(with_tags=True)
docs_plain, _ = load_resume_documents(with_tags=False)
display(df[['candidate_id', 'class']])
print(
    f'выборка: {len(df)} резюме, {df["class"].nunique()} категорий, '
    f'по {int(df.groupby("class").size().min())}–{int(df.groupby("class").size().max())} на класс.'
)
print('документов:', len(docs_tags))
print(docs_tags[0][:400])
print('---')
print(docs_plain[0][:400])


C:\Users\Zhenya\Desktop\МО 3\mo3_team_colabs\labs\ML-HW10\data
True


,candidate_id,class
0,1,Медицина и здравоохранение
1,2,Медицина и здравоохранение
2,3,Медицина и здравоохранение
3,4,Образование и наука
4,5,Образование и наука
5,6,Образование и наука
6,7,Строительство и ремонт
7,8,Строительство и ремонт
8,9,Строительство и ремонт
9,10,Торговля и продажи


выборка: 18 резюме, 6 категорий, по 3–3 на класс.
документов: 18
Кандидат №1
Категория: Медицина и здравоохранение
Расчесывание, стрижка волос взрослых и детей.

Укладка и завивка волос в соответствии с направлением моды и особенностями лица.

Завивка волос на бигуди, специальными зажимами и щипцами, химическим и электрическим способом (перманент).

Массаж и мытье головы.

Нанесение на волосы химических препаратов и растворов.

Окраска волос в различные цвета и
---
Кандидат №1
Расчесывание, стрижка волос взрослых и детей.

Укладка и завивка волос в соответствии с направлением моды и особенностями лица.

Завивка волос на бигуди, специальными зажимами и щипцами, химическим и электрическим способом (перманент).

Массаж и мытье головы.

Нанесение на волосы химических препаратов и растворов.

Окраска волос в различные цвета и оттенки, их обесцвечивание


## 3. FAISS: база с тегами и без тегов

In [3]:
packs = ensure_resume_indices(force=False)
for name, pack in packs.items():
    print(name, 'ntotal =', pack['index'].ntotal)
idx_tags = packs['resumes_with_tags']
idx_plain = packs['resumes_no_tags']


resumes_with_tags ntotal = 18
resumes_no_tags ntotal = 18


## 4. Проверочные вопросы

1. Кто умеет химическую завивку и окраску волос? → кандидат №1  
2. У кого полевые визиты к педиатрам, терапевтам и неврологам? → кандидат №3  
3. Есть ли кандидат с опытом работы поваром? → в базе нет


In [4]:
from questions import HOMEWORK_QUESTIONS

QUESTIONS_HW = HOMEWORK_QUESTIONS
for item in QUESTIONS_HW:
    print(item['kind'], ':', item['q'])
    print('  эталон:', item['gold'])


has_answer : Кто из кандидатов умеет делать химическую завивку и окраску волос?
  эталон: Кандидат №1 (в резюме: завивка, перманент, окраска волос).
has_answer : У какого кандидата есть опыт полевых визитов к педиатрам, терапевтам и неврологам?
  эталон: Кандидат №3 (медицинский промоушен / визиты к врачам).
no_answer : Есть ли в базе кандидат с опытом работы поваром?
  эталон: Повара в выборке нет — модель должна сказать, что информации нет.


## 5. Базовый RAG

In [5]:
BASE = GenParams(temperature=0.1, top_k=50, top_p=0.8, max_new_tokens=100)
DISTANCE_THRESHOLD = 55.0

for item in QUESTIONS_HW:
    out = rag_answer(
        item['q'],
        idx_tags['index'],
        idx_tags['documents'],
        top_k=3,
        distance_threshold=DISTANCE_THRESHOLD,
        params=BASE,
        lang='ru',
    )
    print('Q:', item['q'])
    print('distances:', out['distances'])
    print('relevant:', len(out['relevant']), '| used_llm:', out['used_llm'])
    print('A:', out['answer'])
    print('-' * 60)


W1107 23:06:18.833000 22772 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Q: Кто из кандидатов умеет делать химическую завивку и окраску волос?
distances: [9.457117080688477, 12.038254737854004, 12.626923561096191, 13.132616996765137, 13.142692565917969]
relevant: 5 | used_llm: True
A: Кандидат №1 умеет делать химическую завивку и окраску волос, так как в его опыте указано: «Завивка волос на бигуди, специальными зажимами и щипцами, химическим и электрическим способом (перманент). Окраска волос в различные цвета и оттенки, их обесцвечивание».
------------------------------------------------------------


Q: У какого кандидата есть опыт полевых визитов к педиатрам, терапевтам и неврологам?
distances: [13.440272331237793, 14.353874206542969, 14.36313247680664, 14.455164909362793, 14.555912017822266]
relevant: 5 | used_llm: True
A: 1) У кандидата №3 есть опыт полевых визитов к специалистам, включая педиатров, терапевтов и неврологов, поскольку в его обязанностях прямо указано: «Полевые визиты к специалистам» и перечислены целевые аудитории — педиатры, терапевты, неврологи.
------------------------------------------------------------


Q: Есть ли в базе кандидат с опытом работы поваром?
distances: [9.611769676208496, 10.034481048583984, 10.351226806640625, 10.388532638549805, 10.59630298614502]
relevant: 5 | used_llm: True
A: В базе нет релевантной информации по этому запросу (нет такого кандидата / категории / факта).
------------------------------------------------------------


## 6. Сравнение баз с тегами и без

In [6]:
rows = []
for name, pack in [('with_tags', idx_tags), ('no_tags', idx_plain)]:
    for item in QUESTIONS_HW:
        out = rag_answer(
            item['q'], pack['index'], pack['documents'],
            top_k=3, distance_threshold=DISTANCE_THRESHOLD, params=BASE, lang='ru',
        )
        rows.append({
            'base': name,
            'question': item['q'],
            'n_relevant': len(out['relevant']),
            'used_llm': out['used_llm'],
            'answer': out['answer'],
        })
cmp_df = pd.DataFrame(rows)
display(cmp_df)
same = (
    cmp_df.pivot(index='question', columns='base', values='answer')
    .assign(eq=lambda x: x['with_tags'] == x['no_tags'])['eq'].sum()
)
print(
    f'с тегами и без: совпало ответов {same}/{cmp_df["question"].nunique()}; '
    f'оба варианта находят №1/№3 и отказывают на поваре.'
)


,base,question,n_relevant,used_llm,answer
0,with_tags,Кто из кандидатов умеет делать химическую зави...,5,True,Кандидат №1 умеет делать химическую завивку и ...
1,with_tags,У какого кандидата есть опыт полевых визитов к...,5,True,1) У кандидата №3 есть опыт полевых визитов к ...
2,with_tags,Есть ли в базе кандидат с опытом работы поваром?,5,True,В базе нет релевантной информации по этому зап...
3,no_tags,Кто из кандидатов умеет делать химическую зави...,5,True,Кандидат №1 умеет делать химическую завивку и ...
4,no_tags,У какого кандидата есть опыт полевых визитов к...,5,True,1) У кандидата №3 есть опыт полевых визитов к ...
5,no_tags,Есть ли в базе кандидат с опытом работы поваром?,5,True,В базе нет релевантной информации по этому зап...


с тегами и без: совпало ответов 2/3; оба варианта находят №1/№3 и отказывают на поваре.


## 7. Эксперименты с параметрами

Один параметр за раз. Значения: `top_k_search ∈ {1,3,5,10}`, `temperature ∈ {0.1,0.4,0.7,1.0}` (3 повтора), `top_k_gen ∈ {5,10,50,100}`, `top_p ∈ {0.3,0.5,0.8,1.0}`.


In [7]:
exp_path = CACHE_DIR / 'experiments_resumes.csv'
if not exp_path.exists():
    %run run_experiments.py
exp_df = pd.read_csv(exp_path)
exp_df['ok'] = exp_df['eval'].astype(str).str.startswith('ok')

def family(s):
    s = str(s)
    if s.startswith('baseline'):
        return 'baseline'
    if 'top_k_search' in s:
        return 'top_k_search'
    if 'temperature' in s:
        return 'temperature'
    if 'top_k_gen' in s:
        return 'top_k_gen'
    if 'top_p' in s:
        return 'top_p'
    return 'other'

exp_df['family'] = exp_df['settings'].map(family)
print('строк:', len(exp_df))
display(exp_df['family'].value_counts().to_frame('count'))
print(
    f'сводка прогонов: всего {len(exp_df)}, ok={exp_df["ok"].mean():.0%}, '
    f'семьи = {", ".join(exp_df["family"].value_counts().index)}.'
)
display(exp_df[['dataset', 'question', 'settings', 'n_relevant', 'eval', 'sec', 'answer']])
print(
    f'полная таблица экспериментов: время {exp_df.sec.min():.2f}–{exp_df.sec.max():.2f}s, '
    f'все eval начинаются с ok={exp_df["ok"].all()}.'
)


строк: 54


,count
family,
top_k_search,12
top_k_gen,12
temperature,12
top_p,12
baseline,6


сводка прогонов: всего 54, ok=100%, семьи = top_k_search, top_k_gen, temperature, top_p, baseline.


,dataset,question,settings,n_relevant,eval,sec,answer
0,resumes_with_tags,Кто из кандидатов умеет делать химическую зави...,baseline top_k_search=3 temp=0.1 top_k_gen=50 ...,3,ok: есть ожидаемый факт,1.19,Кандидат №1 умеет делать химическую завивку и ...
1,resumes_with_tags,У какого кандидата есть опыт полевых визитов к...,baseline top_k_search=3 temp=0.1 top_k_gen=50 ...,3,ok: есть ожидаемый факт,1.70,У Кандидата №3 есть опыт полевых визитов к пед...
2,resumes_with_tags,Есть ли в базе кандидат с опытом работы поваром?,baseline top_k_search=3 temp=0.1 top_k_gen=50 ...,3,ok: отказ без галлюцинации,1.51,В базе нет релевантной информации.
3,resumes_with_tags,Кто из кандидатов умеет делать химическую зави...,top_k_search=1 (остальное baseline),1,ok: есть ожидаемый факт,1.39,Кандидат №1 умеет делать химическую завивку и ...
4,resumes_with_tags,У какого кандидата есть опыт полевых визитов к...,top_k_search=1 (остальное baseline),1,ok: есть ожидаемый факт,1.61,У кандидата №3 есть опыт полевых визитов к пед...
5,resumes_with_tags,Есть ли в базе кандидат с опытом работы поваром?,top_k_search=1 (остальное baseline),1,ok: отказ без галлюцинации,0.94,В базе нет релевантной информации.
6,resumes_with_tags,Кто из кандидатов умеет делать химическую зави...,top_k_search=3 (остальное baseline),3,ok: есть ожидаемый факт,1.26,Кандидат №1 умеет делать химическую завивку и ...
7,resumes_with_tags,У какого кандидата есть опыт полевых визитов к...,top_k_search=3 (остальное baseline),3,ok: есть ожидаемый факт,2.42,У Кандидата №3 есть опыт полевых визитов к пед...
8,resumes_with_tags,Есть ли в базе кандидат с опытом работы поваром?,top_k_search=3 (остальное baseline),3,ok: отказ без галлюцинации,0.87,В базе нет релевантной информации.
9,resumes_with_tags,Кто из кандидатов умеет делать химическую зави...,top_k_search=5 (остальное baseline),5,ok: есть ожидаемый факт,1.67,Кандидат №1 умеет делать химическую завивку и ...


полная таблица экспериментов: время 0.84–4.27s, все eval начинаются с ok=True.


### 7.1. Baseline

In [8]:
base_view = exp_df[exp_df['family'] == 'baseline'][
    ['dataset', 'question', 'n_relevant', 'eval', 'sec', 'answer']
].reset_index(drop=True)
display(base_view)
ok_rate = base_view['eval'].astype(str).str.startswith('ok').mean()
print('ok_rate:', round(ok_rate, 3))
print(
    f'baseline: {len(base_view)} строк (2 базы × 3 вопроса), ok_rate={ok_rate:.0%}; '
    f'№1/№3 находятся, повар — отказ.'
)


,dataset,question,n_relevant,eval,sec,answer
0,resumes_with_tags,Кто из кандидатов умеет делать химическую зави...,3,ok: есть ожидаемый факт,1.19,Кандидат №1 умеет делать химическую завивку и ...
1,resumes_with_tags,У какого кандидата есть опыт полевых визитов к...,3,ok: есть ожидаемый факт,1.70,У Кандидата №3 есть опыт полевых визитов к пед...
2,resumes_with_tags,Есть ли в базе кандидат с опытом работы поваром?,3,ok: отказ без галлюцинации,1.51,В базе нет релевантной информации.
3,resumes_no_tags,Кто из кандидатов умеет делать химическую зави...,3,ok: есть ожидаемый факт,1.81,Кандидат №1 умеет делать химическую завивку и ...
4,resumes_no_tags,У какого кандидата есть опыт полевых визитов к...,3,ok: есть ожидаемый факт,2.26,У Кандидата №3 есть опыт полевых визитов к пед...
5,resumes_no_tags,Есть ли в базе кандидат с опытом работы поваром?,3,ok: отказ без галлюцинации,1.98,В базе нет релевантной информации.


ok_rate: 1.0
baseline: 6 строк (2 базы × 3 вопроса), ok_rate=100%; №1/№3 находятся, повар — отказ.


### 7.2. top_k в поиске

In [9]:
tk = exp_df[exp_df['family'] == 'top_k_search'].copy()
tk['top_k_search'] = tk['settings'].str.extract(r'top_k_search=(\d+)').astype(int)
display(tk.sort_values(['top_k_search', 'question'])[
    ['top_k_search', 'question', 'n_relevant', 'eval', 'sec', 'answer']
].reset_index(drop=True))
print('по каждому k три вопроса: смотрим, не ломается ли ответ при узком/широком контексте.')
display(tk.groupby('top_k_search')['ok'].mean().to_frame('ok_rate'))
print(f'ok_rate по k: везде {tk.groupby("top_k_search")["ok"].mean().min():.0%}–{tk.groupby("top_k_search")["ok"].mean().max():.0%}.')
display(tk.groupby('top_k_search')[['n_relevant', 'sec']].mean())
print(
    f'среднее: n_relevant растёт с k ({tk.groupby("top_k_search")["n_relevant"].mean().min():.0f}→'
    f'{tk.groupby("top_k_search")["n_relevant"].mean().max():.0f}), '
    f'время {tk.groupby("top_k_search")["sec"].mean().min():.2f}→'
    f'{tk.groupby("top_k_search")["sec"].mean().max():.2f}s.'
)


,top_k_search,question,n_relevant,eval,sec,answer
0,1,Есть ли в базе кандидат с опытом работы поваром?,1,ok: отказ без галлюцинации,0.94,В базе нет релевантной информации.
1,1,Кто из кандидатов умеет делать химическую зави...,1,ok: есть ожидаемый факт,1.39,Кандидат №1 умеет делать химическую завивку и ...
2,1,У какого кандидата есть опыт полевых визитов к...,1,ok: есть ожидаемый факт,1.61,У кандидата №3 есть опыт полевых визитов к пед...
3,3,Есть ли в базе кандидат с опытом работы поваром?,3,ok: отказ без галлюцинации,0.87,В базе нет релевантной информации.
4,3,Кто из кандидатов умеет делать химическую зави...,3,ok: есть ожидаемый факт,1.26,Кандидат №1 умеет делать химическую завивку и ...
5,3,У какого кандидата есть опыт полевых визитов к...,3,ok: есть ожидаемый факт,2.42,У Кандидата №3 есть опыт полевых визитов к пед...
6,5,Есть ли в базе кандидат с опытом работы поваром?,5,ok: отказ без галлюцинации,1.67,В базе нет релевантной информации.
7,5,Кто из кандидатов умеет делать химическую зави...,5,ok: есть ожидаемый факт,1.67,Кандидат №1 умеет делать химическую завивку и ...
8,5,У какого кандидата есть опыт полевых визитов к...,5,ok: есть ожидаемый факт,2.11,У Кандидата №3 есть опыт полевых визитов к пед...
9,10,Есть ли в базе кандидат с опытом работы поваром?,10,ok: отказ без галлюцинации,2.76,В базе нет релевантной информации.


по каждому k три вопроса: смотрим, не ломается ли ответ при узком/широком контексте.


,ok_rate
top_k_search,
1,1.0
3,1.0
5,1.0
10,1.0


ok_rate по k: везде 100%–100%.


,n_relevant,sec
top_k_search,,
1,1.0,1.313333
3,3.0,1.516667
5,5.0,1.816667
10,10.0,3.446667


среднее: n_relevant растёт с k (1→10), время 1.31→3.45s.


### 7.3. temperature

In [10]:
tp = exp_df[exp_df['family'] == 'temperature'].copy()
tp['temperature'] = tp['settings'].str.extract(r'temperature=([0-9.]+)').astype(float)
tp['repeat'] = tp['settings'].str.extract(r'repeat=(\d+)').astype(int)
display(tp.sort_values(['temperature', 'repeat'])[
    ['temperature', 'repeat', 'eval', 'sec', 'answer']
].reset_index(drop=True))
print('один вопрос (завивка), по 3 повтора на temperature — насколько плывёт формулировка.')
for t, g in tp.groupby('temperature'):
    uniq = g['answer'].drop_duplicates().tolist()
    print(f'temp={t}: {len(uniq)} уникальных из {len(g)}')
    for a in uniq:
        print(' ', a)
nuniq = tp.groupby('temperature')['answer'].nunique()
print(
    f'итог: уникальных формулировок на температуру {int(nuniq.min())}–{int(nuniq.max())} '
    f'из 3 повторов.'
)


,temperature,repeat,eval,sec,answer
0,0.1,1,ok: есть ожидаемый факт,1.28,Кандидат №1 умеет делать химическую завивку и ...
1,0.1,2,ok: есть ожидаемый факт,1.16,Кандидат №1 умеет делать химическую завивку и ...
2,0.1,3,ok: есть ожидаемый факт,1.22,Кандидат №1 умеет делать химическую завивку и ...
3,0.4,1,ok: есть ожидаемый факт,1.20,Кандидат №1 умеет делать химическую завивку и ...
4,0.4,2,ok: есть ожидаемый факт,1.32,Кандидат №1 умеет делать химическую завивку и ...
5,0.4,3,ok: есть ожидаемый факт,1.32,Кандидат №1 умеет делать химическую завивку и ...
6,0.7,1,ok: есть ожидаемый факт,1.29,Кандидат №1 умеет делать химическую завивку и ...
7,0.7,2,ok: есть ожидаемый факт,1.32,Кандидат №1 умеет делать химическую завивку и ...
8,0.7,3,ok: есть ожидаемый факт,1.35,Кандидат №1 умеет делать химическую завивку и ...
9,1.0,1,ok: есть ожидаемый факт,1.29,Кандидат №1 умеет делать химическую завивку и ...


один вопрос (завивка), по 3 повтора на temperature — насколько плывёт формулировка.
temp=0.1: 1 уникальных из 3
  Кандидат №1 умеет делать химическую завивку и окраску волос.
temp=0.4: 1 уникальных из 3
  Кандидат №1 умеет делать химическую завивку и окраску волос.
temp=0.7: 1 уникальных из 3
  Кандидат №1 умеет делать химическую завивку и окраску волос.
temp=1.0: 1 уникальных из 3
  Кандидат №1 умеет делать химическую завивку и окраску волос.
итог: уникальных формулировок на температуру 1–1 из 3 повторов.


### 7.4. top_k в генерации

In [11]:
tg = exp_df[exp_df['family'] == 'top_k_gen'].copy()
tg['top_k_gen'] = tg['settings'].str.extract(r'top_k_gen=(\d+)').astype(int)
display(tg.sort_values(['top_k_gen', 'question'])[
    ['top_k_gen', 'question', 'eval', 'sec', 'answer']
].reset_index(drop=True))
print('меняли только top_k генерации; факты (№1/№3/отказ) должны держаться.')
display(tg.groupby('top_k_gen')['ok'].mean().to_frame('ok_rate'))
print(f'ok_rate по top_k_gen: min={tg.groupby("top_k_gen")["ok"].mean().min():.0%}.')
display(tg.groupby('top_k_gen')['answer'].nunique().to_frame('unique_answers'))
print(
    f'уникальных ответов на значение: {int(tg.groupby("top_k_gen")["answer"].nunique().min())}–'
    f'{int(tg.groupby("top_k_gen")["answer"].nunique().max())} (обычно по числу вопросов).'
)


,top_k_gen,question,eval,sec,answer
0,5,Есть ли в базе кандидат с опытом работы поваром?,ok: отказ без галлюцинации,1.45,В базе нет релевантной информации.
1,5,Кто из кандидатов умеет делать химическую зави...,ok: есть ожидаемый факт,1.28,Кандидат №1 умеет делать химическую завивку и ...
2,5,У какого кандидата есть опыт полевых визитов к...,ok: есть ожидаемый факт,2.03,У Кандидата №3 есть опыт полевых визитов к пед...
3,10,Есть ли в базе кандидат с опытом работы поваром?,ok: отказ без галлюцинации,0.92,В базе нет релевантной информации.
4,10,Кто из кандидатов умеет делать химическую зави...,ok: есть ожидаемый факт,1.29,Кандидат №1 умеет делать химическую завивку и ...
5,10,У какого кандидата есть опыт полевых визитов к...,ok: есть ожидаемый факт,1.82,У Кандидата №3 есть опыт полевых визитов к пед...
6,50,Есть ли в базе кандидат с опытом работы поваром?,ok: отказ без галлюцинации,0.95,В базе нет релевантной информации.
7,50,Кто из кандидатов умеет делать химическую зави...,ok: есть ожидаемый факт,1.43,Кандидат №1 умеет делать химическую завивку и ...
8,50,У какого кандидата есть опыт полевых визитов к...,ok: есть ожидаемый факт,1.96,У Кандидата №3 есть опыт полевых визитов к пед...
9,100,Есть ли в базе кандидат с опытом работы поваром?,ok: отказ без галлюцинации,0.92,В базе нет релевантной информации.


меняли только top_k генерации; факты (№1/№3/отказ) должны держаться.


,ok_rate
top_k_gen,
5,1.0
10,1.0
50,1.0
100,1.0


ok_rate по top_k_gen: min=100%.


,unique_answers
top_k_gen,
5,3
10,3
50,3
100,3


уникальных ответов на значение: 3–3 (обычно по числу вопросов).


### 7.5. top_p

In [12]:
pp = exp_df[exp_df['family'] == 'top_p'].copy()
pp['top_p'] = pp['settings'].str.extract(r'top_p=([0-9.]+)').astype(float)
display(pp.sort_values(['top_p', 'question'])[
    ['top_p', 'question', 'eval', 'sec', 'answer']
].reset_index(drop=True))
print('то же для top_p: влияет ли nucleus sampling на ok и текст.')
display(pp.groupby('top_p')['ok'].mean().to_frame('ok_rate'))
print(f'ok_rate по top_p: min={pp.groupby("top_p")["ok"].mean().min():.0%}.')
display(pp.groupby('top_p')['answer'].nunique().to_frame('unique_answers'))
print(
    f'уникальных ответов на top_p: {int(pp.groupby("top_p")["answer"].nunique().min())}–'
    f'{int(pp.groupby("top_p")["answer"].nunique().max())}.'
)


,top_p,question,eval,sec,answer
0,0.3,Есть ли в базе кандидат с опытом работы поваром?,ok: отказ без галлюцинации,0.95,В базе нет релевантной информации.
1,0.3,Кто из кандидатов умеет делать химическую зави...,ok: есть ожидаемый факт,1.34,Кандидат №1 умеет делать химическую завивку и ...
2,0.3,У какого кандидата есть опыт полевых визитов к...,ok: есть ожидаемый факт,1.89,У Кандидата №3 есть опыт полевых визитов к пед...
3,0.5,Есть ли в базе кандидат с опытом работы поваром?,ok: отказ без галлюцинации,0.93,В базе нет релевантной информации.
4,0.5,Кто из кандидатов умеет делать химическую зави...,ok: есть ожидаемый факт,1.38,Кандидат №1 умеет делать химическую завивку и ...
5,0.5,У какого кандидата есть опыт полевых визитов к...,ok: есть ожидаемый факт,1.91,У Кандидата №3 есть опыт полевых визитов к пед...
6,0.8,Есть ли в базе кандидат с опытом работы поваром?,ok: отказ без галлюцинации,0.86,В базе нет релевантной информации.
7,0.8,Кто из кандидатов умеет делать химическую зави...,ok: есть ожидаемый факт,1.25,Кандидат №1 умеет делать химическую завивку и ...
8,0.8,У какого кандидата есть опыт полевых визитов к...,ok: есть ожидаемый факт,1.72,У Кандидата №3 есть опыт полевых визитов к пед...
9,1.0,Есть ли в базе кандидат с опытом работы поваром?,ok: отказ без галлюцинации,0.84,В базе нет релевантной информации.


то же для top_p: влияет ли nucleus sampling на ok и текст.


,ok_rate
top_p,
0.3,1.0
0.5,1.0
0.8,1.0
1.0,1.0


ok_rate по top_p: min=100%.


,unique_answers
top_p,
0.3,3
0.5,3
0.8,3
1.0,3


уникальных ответов на top_p: 3–3.


### 7.6. С тегами / без тегов

In [13]:
tag_cmp = exp_df[exp_df['family'] == 'baseline'][
    ['dataset', 'question', 'n_relevant', 'eval', 'answer']
].reset_index(drop=True)
display(tag_cmp)
w = tag_cmp[tag_cmp.dataset == 'resumes_with_tags'].set_index('question')['answer']
n = tag_cmp[tag_cmp.dataset == 'resumes_no_tags'].set_index('question')['answer']
same = int((w == n).sum())
ok_t = tag_cmp.loc[tag_cmp.dataset == 'resumes_with_tags', 'eval'].astype(str).str.startswith('ok').mean()
ok_n = tag_cmp.loc[tag_cmp.dataset == 'resumes_no_tags', 'eval'].astype(str).str.startswith('ok').mean()
print(f'теги vs без: совпало формулировок {same}/{len(w)}, ok с тегами={ok_t:.0%}, без={ok_n:.0%}.')


,dataset,question,n_relevant,eval,answer
0,resumes_with_tags,Кто из кандидатов умеет делать химическую зави...,3,ok: есть ожидаемый факт,Кандидат №1 умеет делать химическую завивку и ...
1,resumes_with_tags,У какого кандидата есть опыт полевых визитов к...,3,ok: есть ожидаемый факт,У Кандидата №3 есть опыт полевых визитов к пед...
2,resumes_with_tags,Есть ли в базе кандидат с опытом работы поваром?,3,ok: отказ без галлюцинации,В базе нет релевантной информации.
3,resumes_no_tags,Кто из кандидатов умеет делать химическую зави...,3,ok: есть ожидаемый факт,Кандидат №1 умеет делать химическую завивку и ...
4,resumes_no_tags,У какого кандидата есть опыт полевых визитов к...,3,ok: есть ожидаемый факт,У Кандидата №3 есть опыт полевых визитов к пед...
5,resumes_no_tags,Есть ли в базе кандидат с опытом работы поваром?,3,ok: отказ без галлюцинации,В базе нет релевантной информации.


теги vs без: совпало формулировок 3/3, ok с тегами=100%, без=100%.


## 8. Демонстрация в web

In [14]:
print('Чат: http://127.0.0.1:8000')
print('RAG_BASE=resumes_no_tags, модель=', OLLAMA_MODEL)


Чат: http://127.0.0.1:8000
RAG_BASE=resumes_no_tags, модель= qwen3:4b-instruct


## 9. Выводы

Цифры ниже — из таблиц в разделах 6–7 (54 прогона, модель `qwen3:4b-instruct`, ok = 100%).

По baseline RAG находит нужных людей: завивка/окраска → кандидат №1, визиты к врачам → кандидат №3. На вопрос про повара отвечает отказом («в базе нет релевантной информации») и на базе с тегами, и без — повар не выдумывается. Это же видно по всем 14 прогонам с этим вопросом.

Сравнение `top_k` поиска (1 / 3 / 5 / 10): ok везде 100%, число релевантных документов растёт как k, время с ~1.3s до ~3.5s. Для коротких фактологических вопросов хватает k=3; больший k только раздувает контекст.

Temperature 0.1 / 0.4 / 0.7 / 1.0 (по 3 повтора на одном вопросе): во всех 12 ответах одна и та же формулировка про кандидата №1. На этих вопросах температура почти не меняет текст.

`top_k_gen` (5…100) и `top_p` (0.3…1.0) тоже держат ok=100%. Параметры генерации здесь вторичны относительно retrieval и порога релевантности.

База с тегами vs без тегов: на всех трёх baseline-вопросах ответы совпали, ok 100% в обоих случаях. Для этих фактов теги не обязательны — нужное уже есть в тексте резюме.

Для демо оставляю `top_k_search=3`, `top_k_gen=50`, `top_p≈0.8–0.9`, база `resumes_no_tags` (без категорий в индексе). Время ответов в таблице примерно 0.8–4.3s. Чат: http://127.0.0.1:8000.
